# Investment AI
## Notebook 07 — Build the Research Agent

The retrieval layer is now working.

This notebook gives an LLM-controlled agent access to that retrieval layer so it can:

1. understand a research question,
2. decide when to search,
3. retrieve evidence from Databricks AI Search,
4. reason over the evidence,
5. return an answer with source citations.

```text
User Question
     ↓
Research Agent — Claude Sonnet
     ↓
AI Bridge Retrieval Tool
     ↓
Databricks AI Search
     ↓
Research Evidence
     ↓
Reasoning + Citations
```

The Research Agent is intentionally scoped to **unstructured investment research**.

It does not answer portfolio-position or ownership questions. Those will belong to the Portfolio Agent later.


## Why AI Bridge here?

For new Python agents, Databricks recommends **AI Bridge retriever tools** for querying AI Search indexes directly.

So instead of wiring the agent to the older AI Search workspace MCP endpoint, we expose our existing AI Search index as a Python retrieval tool.

We will use:

- `VectorSearchRetrieverTool` from `databricks-openai`
- OpenAI Agents SDK for the agent loop
- Claude Sonnet on Databricks for reasoning
- Hybrid retrieval as the default search strategy

The AI Search index remains governed by Unity Catalog permissions.


In [0]:
%pip install -q --upgrade databricks-openai openai-agents
dbutils.library.restartPython()


In [0]:
import json

from agents import (
    Agent,
    Runner,
    function_tool,
    set_default_openai_api,
    set_default_openai_client,
)
from agents.tracing import set_trace_processors

from databricks_openai import (
    AsyncDatabricksOpenAI,
    VectorSearchRetrieverTool,
)

AI_SEARCH_INDEX = (
    "investment_ai.research.investment_ai_research_index"
)

RESEARCH_MODEL = "databricks-claude-sonnet-4-5"

#RESEARCH_MODEL = "databricks-llama-4-maverick" 


RETURN_COLUMNS = [
    "chunk_to_retrieve",
    "security_id",
    "company_name",
    "document_type",
    "source_type",
    "source_authority_rank",
    "publish_date",
    "citation_label",
]

print("Research model :", RESEARCH_MODEL)
print("AI Search index:", AI_SEARCH_INDEX)


## 1. Create the research retrieval capability

This is the bridge between the agent and our AI Search index.

The agent does not read the Delta table directly.

Instead, it calls a retrieval tool that searches the governed AI Search index using **Hybrid Search** and returns the most relevant evidence.

Optional metadata filters let the agent narrow retrieval when it already knows the security, quarter, year, or preferred source type.


In [0]:
def _run_research_search(
    query: str,
    security_id: str | None = None,
    fiscal_quarter: str | None = None,
    fiscal_year: int | None = None,
    source_type: str | None = None,
):
    filters = {}

    if security_id:
        filters["security_id"] = security_id

    if fiscal_quarter:
        filters["fiscal_quarter"] = fiscal_quarter

    if fiscal_year:
        filters["fiscal_year"] = fiscal_year

    if source_type:
        filters["source_type"] = source_type

    retriever = VectorSearchRetrieverTool(
        index_name=AI_SEARCH_INDEX,
        tool_name="investment_research_search",
        tool_description=(
            "Search the governed Investment AI research corpus. "
            "Use it for earnings, guidance, outlook, management commentary, "
            "business drivers, risks, and other company-research questions."
        ),
        num_results=5,
        columns=RETURN_COLUMNS,
        filters=filters or None,
        query_type="HYBRID",
    )

    return retriever.execute(query=query)


print("✅ Research retrieval capability created.")


## 2. Test retrieval before giving it to the agent

We already tested AI Search directly in Notebook 06.

Here we make one quick call through the same retrieval abstraction the agent will use.

This keeps retrieval and reasoning separate:

```text
Retriever → finds evidence
Agent     → reasons over evidence
```


In [0]:
retrieval_preview = _run_research_search(
    query="Why was the full-year outlook reduced?",
    security_id="IA-SYN-01",
    fiscal_quarter="Q2",
    fiscal_year=2026,
    source_type="primary",
)

print(retrieval_preview)


## 3. Expose retrieval as an agent tool

The `@function_tool` decorator makes our Python retrieval function available to the agent.

The model can now decide:

- whether a search is needed,
- what query to send,
- whether useful metadata filters are known,
- how to use the returned evidence in the final answer.


In [0]:
@function_tool
def search_investment_research(
    query: str,
    security_id: str | None = None,
    fiscal_quarter: str | None = None,
    fiscal_year: int | None = None,
    source_type: str | None = None,
) -> str:
    """Search the governed synthetic investment research corpus.

    Use this tool for company outlook, guidance, earnings commentary,
    business drivers, risks, management statements, and research evidence.
    """

    results = _run_research_search(
        query=query,
        security_id=security_id,
        fiscal_quarter=fiscal_quarter,
        fiscal_year=fiscal_year,
        source_type=source_type,
    )

    return json.dumps(
        results,
        default=str,
    )


print("✅ AI Search is now available as an agent tool.")


## 4. Configure Claude Sonnet for agent reasoning

Claude Sonnet is used here because the Research Agent needs to:

- interpret open-ended research questions,
- decide when and how to retrieve evidence,
- compare multiple sources,
- synthesize explanations,
- preserve citations,
- avoid unsupported conclusions.

The Databricks OpenAI-compatible client lets the OpenAI Agents SDK use Claude without changing the agent framework.


In [0]:
set_default_openai_client(
    AsyncDatabricksOpenAI()
)

set_default_openai_api(
    "chat_completions"
)

# We will introduce MLflow tracing explicitly later in the project.
set_trace_processors([])

print("✅ Claude Sonnet client configured.")


## 5. Define the Research Agent

The instructions are part of the agent's behavior contract.

A few rules are especially important:

1. Use retrieval for factual company-research claims.
2. Prefer primary evidence when sources conflict.
3. Cite the retrieved source labels.
4. Do not invent facts that were not retrieved.
5. Stay within research scope.
6. Do not answer portfolio ownership or exposure questions.

That last boundary becomes important when we introduce the Portfolio Agent and Supervisor later.


In [0]:
RESEARCH_AGENT_INSTRUCTIONS = """
You are the Investment AI Research Agent.

Your responsibility is unstructured investment research only.

Use the search_investment_research tool whenever the user asks about:
- company guidance or outlook,
- earnings commentary,
- reasons for business changes,
- company risks or drivers,
- management statements,
- research evidence.

Evidence rules:
- Ground factual research claims in retrieved evidence.
- Prefer primary sources over internal sources, and internal sources over
  secondary sources when evidence conflicts.
- Use citation_label from the retrieved results when citing evidence.
- Do not invent a citation or fact that was not returned by the tool.
- If evidence is incomplete or conflicting, say so clearly.

Scope boundary:
- You do NOT have portfolio holdings, ownership, exposure, or fund-position data.
- If the user asks how much Investment AI owns, explain that portfolio data
  belongs to the Portfolio Agent and should be handled separately.

Keep answers concise, analytical, and evidence-based.
"""

research_agent = Agent(
    name="Investment AI Research Agent",
    instructions=RESEARCH_AGENT_INSTRUCTIONS,
    model=RESEARCH_MODEL,
    tools=[search_investment_research],
)

print("✅ Research Agent created.")


## 6. Ask a focused research question

Now the model is no longer being handed retrieved chunks manually.

It receives the user question and decides when to call the AI Search tool.

This is the key transition from:

```text
RAG pipeline
```

to:

```text
tool-using Research Agent
```


In [0]:
question = (
    "Why did Synthetic Compute Company 01 "
    "lower its outlook in Q2 2026?"
)

result = await Runner.run(
    research_agent,
    question,
)

print(result.final_output)


## 7. Ask a broader research question

The Research Agent can also search across the corpus and synthesize evidence from multiple companies.

This is the research-only portion of our flagship business question.


In [0]:
question = (
    "Which companies lowered their outlook in Q2 2026, "
    "and why did they lower it? "
    "Use the strongest available research evidence."
)

result = await Runner.run(
    research_agent,
    question,
)

print(result.final_output)


## 8. Verify the agent boundary

The Research Agent should know what it **cannot** answer.

It can determine:

```text
Which companies lowered outlook?
Why?
What evidence supports that conclusion?
```

It should not determine:

```text
How many shares do we own?
What is our market-value exposure?
Which funds hold the company?
```

Those questions require the structured portfolio layer and will be handled by a separate Portfolio Agent.


In [0]:
boundary_question = (
    "How much do we currently own of the companies "
    "that lowered their outlook?"
)

boundary_result = await Runner.run(
    research_agent,
    boundary_question,
)

print(boundary_result.final_output)


## Research Agent complete

We now have our first specialized agent:

```text
User
 ↓
Research Agent — Claude Sonnet
 ↓
AI Search retrieval tool
 ↓
Hybrid semantic + keyword search
 ↓
Governed research evidence
 ↓
Evidence-based answer + citations
```

The important design principle remains:

> Specialized agents should have the tools and data required for their responsibility — not access to everything.

The Research Agent owns the **unstructured research problem**.

Next, we move to the structured analytics side of the architecture:

**Business Semantics → Metric Views → Genie → Genie Ontology**
